In [1]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
data = np.load("pems08.npz")["data"]

traffic = data[:, :, 0]

print(traffic.shape)

(17856, 170)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    - sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(17844, 12, 170)
(17844, 170)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
import torch
import torch.nn as nn

class TransformerModel(nn.Module):

    def __init__(
        self,
        num_nodes=170,
        d_model=128,
        nhead=4,
        num_layers=2
    ):

        super().__init__()

        self.input_proj = nn.Linear(
            num_nodes,
            d_model
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(
            d_model,
            num_nodes
        )

    def forward(self,x):

        x = self.input_proj(x)

        x = self.transformer(x)

        x = x[:,-1,:]

        x = self.fc(x)

        return x

In [9]:
model = TransformerModel()

X_batch, y_batch = next(iter(train_loader))

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 170])
torch.Size([64, 170])


In [10]:
model = TransformerModel()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(pred, y_batch)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs} "
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.024624
Epoch 2/30 Loss: 0.005943
Epoch 3/30 Loss: 0.004650
Epoch 4/30 Loss: 0.004037
Epoch 5/30 Loss: 0.003707
Epoch 6/30 Loss: 0.003454
Epoch 7/30 Loss: 0.003262
Epoch 8/30 Loss: 0.003104
Epoch 9/30 Loss: 0.002964
Epoch 10/30 Loss: 0.002867
Epoch 11/30 Loss: 0.002834
Epoch 12/30 Loss: 0.002689
Epoch 13/30 Loss: 0.002655
Epoch 14/30 Loss: 0.002628
Epoch 15/30 Loss: 0.002559
Epoch 16/30 Loss: 0.002482
Epoch 17/30 Loss: 0.002471
Epoch 18/30 Loss: 0.002407
Epoch 19/30 Loss: 0.002390
Epoch 20/30 Loss: 0.002349
Epoch 21/30 Loss: 0.002338
Epoch 22/30 Loss: 0.002315
Epoch 23/30 Loss: 0.002312
Epoch 24/30 Loss: 0.002279
Epoch 25/30 Loss: 0.002256
Epoch 26/30 Loss: 0.002229
Epoch 27/30 Loss: 0.002234
Epoch 28/30 Loss: 0.002214
Epoch 29/30 Loss: 0.002202
Epoch 30/30 Loss: 0.002169


In [11]:
train_time= time.time() - train_start

print("Time Taken:", train_time)

Time Taken: 581.6642909049988


In [12]:
torch.save(
    model.state_dict(),
    "Transformer-PEMS08.pth"
)

In [13]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import numpy as np


test_dataset = TensorDataset(
    X_test,
    y_test
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

all_predictions = []
all_targets = []

model.eval()

infer_start = time.time()

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        pred = model(X_batch)

        all_predictions.append(
            pred.numpy()
        )

        all_targets.append(
            y_batch.numpy()
        )

predictions = np.concatenate(
    all_predictions,    
    axis=0
)

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = np.concatenate(
    all_targets,
    axis=0
)

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 0.2774059772491455
MAE: 0.03932343
RMSE: 0.0552145


In [14]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)
print("MAPE:", mape)
print("R2:", r2)

MAPE: 15188.1591796875
R2: 0.9507750655425059


In [15]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 1229866
